# Figure 3: Reach-level slope repeatability comparison

This notebook compares annual temporal repeatability among filtered PIXC slopes, RiverSP `slope`, and RiverSP `slope2`. RMSE is calculated for each reach from deviations about that product's annual median, using only reach-overpasses with all three estimates available.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FixedLocator, FuncFormatter, LogLocator, NullFormatter

INK = "#17222B"
MUTED = "#647078"
GRID = "#DCE2E5"
RIVERSP = "#69757B"
SLOPE2 = "#C17C32"
FILTERED = "#1976A8"
IMPROVE = "#278F86"
WORSE = "#C65A64"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.labelcolor": INK,
    "axes.edgecolor": INK,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 160,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "raqw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
MATCHED_CSV = (
    PROJECT_ROOT / "data" / "riversp_comparison_updated_filter"
    / "matched_pixc_riversp_slopes.csv"
)
FIGURE_DIR = PROJECT_ROOT / "publication_figs" / "outputs"
OUTPUT_STEM = "figure_03_slope_repeatability_comparison"
EXPORT_FILES = False
if EXPORT_FILES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

assert MATCHED_CSV.exists(), MATCHED_CSV

In [ ]:
observations = pd.read_csv(MATCHED_CSV, dtype={"reach_id": str})
columns = {
    "RiverSP slope": "riversp_slope_m_per_km",
    "RiverSP slope2": "riversp_slope2_m_per_km",
    "Filtered PIXC": "updated_postfilter_slope_m_per_km",
}

# A common observation set keeps every reach-level comparison strictly paired.
paired = observations.dropna(subset=list(columns.values())).copy()
rows = []
for reach_id, group in paired.groupby("reach_id", sort=False):
    row = {"reach_id": reach_id, "n_observations": len(group)}
    for label, column in columns.items():
        values = group[column].to_numpy(float)
        annual_median = np.median(values)
        key = label.lower().replace(" ", "_")
        row[f"{key}_median"] = annual_median
        row[f"{key}_rmse"] = np.sqrt(np.mean((values - annual_median) ** 2))
    rows.append(row)

reach_metrics = pd.DataFrame(rows)
rmse_columns = {
    "RiverSP slope": "riversp_slope_rmse",
    "RiverSP slope2": "riversp_slope2_rmse",
    "Filtered PIXC": "filtered_pixc_rmse",
}
reach_metrics["slope_minus_filtered"] = (
    reach_metrics["riversp_slope_rmse"] - reach_metrics["filtered_pixc_rmse"]
)
reach_metrics["slope2_minus_filtered"] = (
    reach_metrics["riversp_slope2_rmse"] - reach_metrics["filtered_pixc_rmse"]
)
reach_metrics["slope_minus_slope2"] = (
    reach_metrics["riversp_slope_rmse"] - reach_metrics["riversp_slope2_rmse"]
)

n_reaches = len(reach_metrics)
median_rmse = {label: reach_metrics[column].median() for label, column in rmse_columns.items()}
pct_better_slope = 100 * (reach_metrics["slope_minus_filtered"] > 0).mean()
pct_better_slope2 = 100 * (reach_metrics["slope2_minus_filtered"] > 0).mean()
pct_slope2_better_slope = 100 * (reach_metrics["slope_minus_slope2"] > 0).mean()

print(f"Common valid reach-overpasses: {len(paired):,}")
print(f"Reaches included: {n_reaches:,}")
print(f"Observations per reach: {reach_metrics['n_observations'].min()}–{reach_metrics['n_observations'].max()} (median {reach_metrics['n_observations'].median():.0f})")
for label, value in median_rmse.items():
    print(f"Median RMSE, {label}: {value:.3f} m km^-1")
print(f"Filtered PIXC lower than RiverSP slope: {pct_better_slope:.1f}% of reaches")
print(f"Filtered PIXC lower than RiverSP slope2: {pct_better_slope2:.1f}% of reaches")
print(f"RiverSP slope2 lower than RiverSP slope: {pct_slope2_better_slope:.1f}% of reaches")

In [ ]:
def panel_heading(ax, letter, title):
    ax.text(
        0, 1.025, f"{letter}   {title}", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=INK,
    )


def clean_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)


method_labels = ["RiverSP\n$slope$", "RiverSP\n$slope2$", "Filtered\nPIXC"]
method_names = ["RiverSP slope", "RiverSP slope2", "Filtered PIXC"]
method_colors = [RIVERSP, SLOPE2, FILTERED]
rmse = reach_metrics[[rmse_columns[name] for name in method_names]].to_numpy(float)

fig = plt.figure(figsize=(10.6, 4.9), facecolor="white")
grid = fig.add_gridspec(
    1, 2, width_ratios=[1.08, 0.92],
    left=0.075, right=0.985, bottom=0.15, top=0.92, wspace=0.27,
)
ax_pair = fig.add_subplot(grid[0, 0])
ax_diff = fig.add_subplot(grid[0, 1])

# Panel A: paired reach-level RMSE values.
x = np.arange(3)
for row in rmse:
    ax_pair.plot(x, row, color="#AAB2B6", linewidth=0.45, alpha=0.18, zorder=1)
for i, color in enumerate(method_colors):
    ax_pair.scatter(
        np.full(n_reaches, i), rmse[:, i], s=9, color=color,
        alpha=0.36, edgecolor="none", rasterized=True, zorder=2,
    )

medians = np.array([median_rmse[name] for name in method_names])
ax_pair.plot(x, medians, color=INK, linewidth=1.5, zorder=5)
for i, (value, color) in enumerate(zip(medians, method_colors)):
    ax_pair.scatter(i, value, s=70, marker="D", color=color, edgecolor="white", linewidth=0.8, zorder=6)
    ax_pair.annotate(
        f"{value:.3f}", (i, value), xytext=(0, -14), textcoords="offset points",
        ha="center", va="top", fontsize=7.8, fontweight="bold", color=color,
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.8, "alpha": 0.88},
    )

ax_pair.set_yscale("log")
ax_pair.set_xlim(-0.35, 2.35)
ax_pair.set_xticks(x, method_labels)
ax_pair.set_ylabel("Reach-level RMSE (m km$^{-1}$)")
ax_pair.yaxis.set_major_locator(LogLocator(base=10))
ax_pair.yaxis.set_minor_formatter(NullFormatter())
ax_pair.grid(axis="y", which="major", color=GRID, linewidth=0.65)
ax_pair.grid(axis="y", which="minor", color=GRID, linewidth=0.35, alpha=0.45)
panel_heading(ax_pair, "a", "Paired reach-level annual RMSE")
clean_axis(ax_pair)

# Panel B: signed RMSE differences. Positive values favor the second estimate named.
difference_specs = [
    ("slope_minus_slope2", "RiverSP $slope$\nminus RiverSP $slope2$", pct_slope2_better_slope, SLOPE2, "$slope2$ lower RMSE"),
    ("slope_minus_filtered", "RiverSP $slope$\nminus filtered PIXC", pct_better_slope, FILTERED, "Filtered lower RMSE"),
    ("slope2_minus_filtered", "RiverSP $slope2$\nminus filtered PIXC", pct_better_slope2, FILTERED, "Filtered lower RMSE"),
]
rng = np.random.default_rng(42)
for y, (column, label, percent, color, annotation) in enumerate(difference_specs):
    values = reach_metrics[column].to_numpy(float)
    jitter = rng.uniform(-0.14, 0.14, len(values))
    point_colors = np.where(values > 0, IMPROVE, WORSE)
    ax_diff.scatter(
        values, y + jitter, s=11, c=point_colors, alpha=0.38,
        edgecolor="none", rasterized=True, zorder=2,
    )
    quartiles = np.quantile(values, [0.25, 0.5, 0.75])
    ax_diff.plot([quartiles[0], quartiles[2]], [y, y], color=INK, linewidth=4.5, solid_capstyle="round", zorder=4)
    median_color = IMPROVE if quartiles[1] > 0 else WORSE
    ax_diff.scatter(quartiles[1], y, s=32, marker="D", color=median_color, edgecolor="white", linewidth=0.7, zorder=5)
    ax_diff.text(
        0.97, y, f"{percent:.1f}%",
        transform=ax_diff.get_yaxis_transform(), ha="right", va="center",
        fontsize=7.4, fontweight="bold", color=MUTED,
    )

ax_diff.axvspan(0.0, 50, color=IMPROVE, alpha=0.055, linewidth=0, zorder=0)
ax_diff.axvline(0, color=INK, linewidth=0.9, zorder=3)
ax_diff.set_xscale("symlog", linthresh=0.05, linscale=0.9)
ax_diff.set_xlim(-50, 20)
difference_ticks = [-10, -1, -0.1, 0, 0.1, 1, 10]
ax_diff.xaxis.set_major_locator(FixedLocator(difference_ticks))
ax_diff.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
ax_diff.set_yticks([0, 1, 2], [spec[1] for spec in difference_specs])
ax_diff.tick_params(axis="y", labelsize=7.6, pad=4)
for label in ax_diff.get_yticklabels():
    label.set_horizontalalignment("right")
    label.set_linespacing(1.15)
ax_diff.set_ylim(-0.48, 2.52)
ax_diff.invert_yaxis()
ax_diff.set_xlabel("RMSE difference (m km$^{-1}$)")
ax_diff.grid(axis="x", color=GRID, linewidth=0.55)
panel_heading(ax_diff, "b", "RMSE difference distributions")
clean_axis(ax_diff)

if EXPORT_FILES:
    png_path = FIGURE_DIR / f"{OUTPUT_STEM}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}.pdf"
    fig.savefig(png_path, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")

plt.show()

In [ ]:
# Exploratory diagnostic only; this cell is not part of the official Figure 3.
comparisons = [
    {
        "x_column": "riversp_slope_rmse",
        "y_column": "riversp_slope2_rmse",
        "x_label": "RiverSP $slope$",
        "y_label": "RiverSP $slope2$",
        "x_color": RIVERSP,
        "y_color": SLOPE2,
    },
    {
        "x_column": "riversp_slope_rmse",
        "y_column": "filtered_pixc_rmse",
        "x_label": "RiverSP $slope$",
        "y_label": "Filtered PIXC",
        "x_color": RIVERSP,
        "y_color": FILTERED,
    },
    {
        "x_column": "riversp_slope2_rmse",
        "y_column": "filtered_pixc_rmse",
        "x_label": "RiverSP $slope2$",
        "y_label": "Filtered PIXC",
        "x_color": SLOPE2,
        "y_color": FILTERED,
    },
]

all_rmse = reach_metrics[[
    "riversp_slope_rmse", "riversp_slope2_rmse", "filtered_pixc_rmse"
]].to_numpy(float)
log_limits = [all_rmse.min() * 0.75, all_rmse.max() * 1.25]

fig_explore, axes = plt.subplots(2, 3, figsize=(12.0, 7.0))
fig_explore.subplots_adjust(
    left=0.07, right=0.985, bottom=0.10, top=0.89,
    wspace=0.28, hspace=0.35,
)
rng_explore = np.random.default_rng(7)

for index, spec in enumerate(comparisons):
    ax_scatter = axes[0, index]
    ax_difference = axes[1, index]
    x_values = reach_metrics[spec["x_column"]].to_numpy(float)
    y_values = reach_metrics[spec["y_column"]].to_numpy(float)
    difference = x_values - y_values
    y_lower = difference > 0

    # Top row: paired reach-level RMSE. Points below the 1:1 line favor the y-axis product.
    ax_scatter.scatter(
        x_values[~y_lower], y_values[~y_lower], s=16,
        color=spec["x_color"], alpha=0.48, edgecolor="none",
        label=f'{spec["x_label"]} lower', rasterized=True,
    )
    ax_scatter.scatter(
        x_values[y_lower], y_values[y_lower], s=16,
        color=spec["y_color"], alpha=0.48, edgecolor="none",
        label=f'{spec["y_label"]} lower', rasterized=True,
    )
    ax_scatter.plot(
        log_limits, log_limits, color=INK, linewidth=0.9,
        linestyle=(0, (4, 3)), label="1:1",
    )
    ax_scatter.set_xscale("log")
    ax_scatter.set_yscale("log")
    ax_scatter.set_xlim(log_limits)
    ax_scatter.set_ylim(log_limits)
    ax_scatter.set_xlabel(f'{spec["x_label"]} RMSE (m km$^{{-1}}$)')
    ax_scatter.set_ylabel(f'{spec["y_label"]} RMSE (m km$^{{-1}}$)')
    ax_scatter.grid(color=GRID, linewidth=0.5, which="major")
    ax_scatter.legend(loc="upper left", frameon=False, fontsize=6.7)
    ax_scatter.set_title(
        f'{chr(97 + index)}   {spec["x_label"]} vs {spec["y_label"]}',
        loc="left", fontsize=9.5, fontweight="bold",
    )
    clean_axis(ax_scatter)

    # Bottom row: x RMSE minus y RMSE. Positive values favor the y-axis product.
    jitter = rng_explore.uniform(-0.17, 0.17, len(difference))
    point_colors = np.where(y_lower, spec["y_color"], spec["x_color"])
    ax_difference.axvspan(0, 50, color=spec["y_color"], alpha=0.055, linewidth=0)
    ax_difference.scatter(
        difference, jitter, s=13, c=point_colors,
        alpha=0.38, edgecolor="none", rasterized=True,
    )
    quartiles = np.quantile(difference, [0.25, 0.5, 0.75])
    ax_difference.plot(
        [quartiles[0], quartiles[2]], [0, 0], color=INK,
        linewidth=4.5, solid_capstyle="round", zorder=4,
    )
    ax_difference.scatter(
        quartiles[1], 0, s=38, marker="D", color=spec["y_color"],
        edgecolor="white", linewidth=0.7, zorder=5,
    )
    ax_difference.axvline(0, color=INK, linewidth=0.9)
    ax_difference.set_xscale("symlog", linthresh=0.05, linscale=0.9)
    ax_difference.set_xlim(-50, 50)
    ax_difference.xaxis.set_major_locator(FixedLocator([-10, -1, -0.1, 0, 0.1, 1, 10]))
    ax_difference.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax_difference.set_ylim(-0.46, 0.46)
    ax_difference.set_yticks([])
    ax_difference.set_xlabel(
        f'RMSE difference: {spec["x_label"]} minus {spec["y_label"]}\n(m km$^{{-1}}$)'
    )
    ax_difference.grid(axis="x", color=GRID, linewidth=0.5)
    ax_difference.text(
        0.97, 0.91,
        f'{spec["y_label"]} lower for {y_lower.mean():.1%}\n'
        f'Median difference = {np.median(difference):.3f}',
        transform=ax_difference.transAxes, ha="right", va="top",
        fontsize=7.5, color=spec["y_color"],
    )
    ax_difference.text(
        0.98, 0.04, f'Positive values favor {spec["y_label"]}',
        transform=ax_difference.transAxes, ha="right", va="bottom",
        fontsize=6.8, color=MUTED,
    )
    ax_difference.set_title(
        f'{chr(100 + index)}   Signed RMSE difference',
        loc="left", fontsize=9.5, fontweight="bold",
    )
    clean_axis(ax_difference)

fig_explore.suptitle(
    "Exploratory pairwise comparison of reach-level RMSE",
    fontsize=11.5, fontweight="bold", color=INK,
)
plt.show()

for spec in comparisons:
    difference = reach_metrics[spec["x_column"]] - reach_metrics[spec["y_column"]]
    print(
        f'{spec["y_label"]} lower than {spec["x_label"]}: '
        f'{(difference > 0).sum():,}/{len(difference):,} reaches '
        f'({(difference > 0).mean():.1%}); median difference = {difference.median():.3f} m km^-1'
    )

**Draft caption.** Figure 3. Reach-level comparison of slope repeatability among RiverSP *slope*, RiverSP *slope2*, and filtered PIXC. (a) Annual reach-level RMSE for each slope estimate, calculated about each product's annual reach median using 7,149 common valid reach-overpasses across 347 reaches. Thin lines connect estimates for the same reach; diamonds and labels show population medians. (b) Paired RMSE difference distributions. Positive values favor the second estimate named in each row. Percentages indicate the fraction of reaches for which the second estimate has lower RMSE. RiverSP *slope2* has lower RMSE than RiverSP *slope* for 68.9% of reaches, filtered PIXC has lower RMSE than RiverSP *slope* for 55.3%, and filtered PIXC has lower RMSE than RiverSP *slope2* for 45.0%. Filtered PIXC has the lowest population-median RMSE, but reach-level performance is heterogeneous.